In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable 

In [0]:
dbutils.widgets.text("incremental_flag","0")

In [0]:
incremental_flag = dbutils.widgets.get("incremental_flag")

In [0]:
df_src = spark.sql('''
                   Select DISTINCT(Dealer_ID) as Dealer_ID, DealerName from 
                   parquet.`abfss://silver@slcardatalake.dfs.core.windows.net/carsales/`
                   ''')

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    df_sink = spark.sql('''
                        Select dim_dealer_key, Dealer_ID, DealerName
                        from cars_catalog.gold.dim_dealer
                        ''')
else:
    df_sink = spark.sql('''
                        Select 1 as dim_dealer_key, Dealer_ID, DealerName
                        from parquet.`abfss://silver@slcardatalake.dfs.core.windows.net/carsales/` where 1=0
                        ''')


In [0]:
df_filter = df_src.join(df_sink, df_src.Dealer_ID == df_sink.Dealer_ID, 'left')\
    .select(df_src['Dealer_ID'],df_src['DealerName'],df_sink['dim_dealer_key'])

In [0]:
df_filter_new = df_filter.filter(col("dim_dealer_key").isNull())

In [0]:
df_filter_old = df_filter.filter(col("dim_dealer_key").isNotNull())

In [0]:
if (incremental_flag == '0'):
    max_value = 1
else:
    max_value_df = spark.sql("Select max(dim_dealer_key) from cars_catalog.gold.dim_dealer")
    max_value = max_value_df.collect()[0][0] + 1

In [0]:
df_filter_new =df_filter_new.withColumn('dim_dealer_key', max_value + monotonically_increasing_id())

In [0]:
df_final = df_filter_new.union(df_filter_old)

In [0]:
if spark.catalog.tableExists('cars_catalog.gold.dim_dealer'):
    delta_tbl = DeltaTable.forPath(spark, 'abfss://gold@slcardatalake.dfs.core.windows.net/dim_dealer')
    delta_tbl.alias('trg').merge(df_final.alias('src'),'trg.dim_dealer_key=src.dim_dealer_key').whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    df_final.write.format('delta').mode('overwrite').\
    option('path','abfss://gold@slcardatalake.dfs.core.windows.net/dim_dealer')\
        .saveAsTable('cars_catalog.gold.dim_dealer')

In [0]:
%sql
Select * from cars_catalog.gold.dim_dealer